# 1.5 构建、正确性与历史结果

## 本节目标

- 按原 README 命令构建和运行 benchmark
- 读取完整历史 CSV
- 将性能趋势与实现证据分开

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


## 构建与快速运行

原工程使用标准 CMake/C++17/OpenMP，生成 `build/bin/spmv_benchmark` 和三个矩阵生成器。在昇腾环境（先 `source` CANN `set_env.sh`）下，下面的命令构建并运行真实 Ascend C RTC 后端（日志 `Actual Backend=Ascend C RTC`）；缺少 CANN/ACL 时 CMake 直接失败是正确行为，可用 `-DSPMV_REAL_ASCENDC=OFF` 做 Host 原型诊断，但该模式结果不是 Ascend 实测。

In [ ]:
%%bash
set -e
cd src/ascend_spmv
bash scripts/build.sh
bash scripts/run.sh --matrix U1 --warmup 1 --repeat 3 --csv results/course_quick_check.csv

## 读取原 README 历史记录

运行时生成的 `spmv_benchmark.csv` 包含 U/L/B 六矩阵、CPU single/OpenMP16、FP32/FP16/BF16/persistent 字段。结果文件不随仓库提交；报告应记录实际环境，并给出最终 BF16 persistent 对 CPU single 的加速和误差。

源码核对表明当前仓库已有真实 Device API：`AscendCSpmvBackend` 的 RTC 链（CMake 默认 `SPMV_REAL_ASCENDC=ON`），运行输出 `Actual Backend=Ascend C RTC`、CSV 字段 `actual_backend=ascend_c`。原 README 的历史 CSV 是旧 Host 模拟实现留下的实验记录，不能冒充当前 Kernel 结果。

## 结果分析

原 README 历史记录（旧 Host 实现）中 U2/L2 的 persistent 路径相对 CPU single 超过 5x，并接近 OpenMP16；小矩阵上 OpenMP16 更快。有效结论是规模、访存、精度和固定成本共同决定最优后端，而不是“NPU 必然更快”。这些数字只代表历史 CSV 中的旧实现，不是当前 Device Kernel 实测。

## 课后实践

选择 U1/U2，分别比较 cold start、warm total、CPU16 和误差，并写出哪些结论来自 CSV、哪些能由当前源码证明。参考答案见 `answer/01.05_answer.md`。